# Bias in Bios Dataset Creation

Creates the sampled Bias-in-Bios dataset and labels the selected biographies with the concept questions.


## Sampling


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_selection import mutual_info_classif

RANDOM_SEED = 13
DATASET_NAME = "LabHC/bias_in_bios"
TEXT_COLUMN = "hard_text"
TARGET_COLUMN = "profession"
SENSITIVE_COLUMN = "gender"

SAMPLE_SIZES = {
    "train": 10_000,
    "validation": 1_000,
    "test": 2_000,
}

TARGET_GENDER_PROFESSION_MI = 0.16
GENDER_AMPLIFICATION_GRID = [1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 4.0]

PROFESSION_NAMES = [
    "accountant",
    "architect",
    "attorney",
    "chiropractor",
    "comedian",
    "composer",
    "dentist",
    "dietitian",
    "dj",
    "filmmaker",
    "interior_designer",
    "journalist",
    "model",
    "nurse",
    "painter",
    "paralegal",
    "pastor",
    "personal_trainer",
    "photographer",
    "physician",
    "poet",
    "professor",
    "psychologist",
    "rapper",
    "software_engineer",
    "surgeon",
    "teacher",
    "yoga_teacher",
]

GENDER_NAMES = {
    0: "male",
    1: "female",
}


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    return start_path


repo_root = find_repo_root(Path.cwd())
data_dir = repo_root / "data" / "bias_in_bios_sample"
data_dir.mkdir(parents=True, exist_ok=True)

{
    "repo_root": str(repo_root),
    "dataset_name": DATASET_NAME,
    "sample_sizes": SAMPLE_SIZES,
    "random_seed": RANDOM_SEED,
    "data_dir": str(data_dir),
}

In [ ]:
raw_splits = {
    "train": load_dataset(DATASET_NAME, split="train"),
    "validation": load_dataset(DATASET_NAME, split="dev"),
    "test": load_dataset(DATASET_NAME, split="test"),
}

{
    split: {
        "rows": len(dataset),
        "columns": dataset.column_names,
    }
    for split, dataset in raw_splits.items()
}

In [ ]:
def clean_biography_texts(texts):
    return (
        texts.astype(str)
        .str.replace("\u2028", " ", regex=False)
        .str.replace("\u2029", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.replace("\n", " ", regex=False)
    )


def dataset_to_frame(dataset, split_name):
    frame = dataset.to_pandas()
    frame = frame[[TEXT_COLUMN, TARGET_COLUMN, SENSITIVE_COLUMN]].copy()
    frame[TEXT_COLUMN] = clean_biography_texts(frame[TEXT_COLUMN])
    frame["source_split"] = split_name
    frame["profession_name"] = frame[TARGET_COLUMN].map(lambda i: PROFESSION_NAMES[int(i)])
    frame["gender_name"] = frame[SENSITIVE_COLUMN].map(lambda i: GENDER_NAMES[int(i)])
    frame["text_length"] = frame[TEXT_COLUMN].str.len()
    return frame


full_frames = {
    split: dataset_to_frame(dataset, split)
    for split, dataset in raw_splits.items()
}

{split: frame.shape for split, frame in full_frames.items()}

In [ ]:
def gender_profession_mutual_info(frame):
    gender = frame[[SENSITIVE_COLUMN]].to_numpy()
    profession = frame[TARGET_COLUMN].to_numpy()
    return float(
        mutual_info_classif(
            gender,
            profession,
            discrete_features=True,
            random_state=RANDOM_SEED,
        )[0]
    )


def balanced_profession_counts(total_rows, profession_labels):
    base_count = total_rows // len(profession_labels)
    remainder = total_rows % len(profession_labels)
    return {
        profession: base_count + int(position < remainder)
        for position, profession in enumerate(profession_labels)
    }


def amplified_gender_target_share(source_female_share, overall_female_share, amplification):
    target = overall_female_share + amplification * (source_female_share - overall_female_share)
    return float(np.clip(target, 0.0, 1.0))


def gender_counts_for_profession(profession_frame, profession_n, overall_female_share, amplification):
    female_available = int(profession_frame[SENSITIVE_COLUMN].eq(1).sum())
    male_available = int(profession_frame[SENSITIVE_COLUMN].eq(0).sum())
    source_female_share = female_available / len(profession_frame)
    target_female_share = amplified_gender_target_share(
        source_female_share,
        overall_female_share,
        amplification,
    )

    female_n = int(round(profession_n * target_female_share))
    female_n = min(max(female_n, 0), female_available)
    male_n = profession_n - female_n
    if male_n > male_available:
        male_n = male_available
        female_n = profession_n - male_n
    if female_n > female_available:
        raise ValueError(
            f"Cannot sample {profession_n} rows from {profession_frame['profession_name'].iloc[0]} "
            "with the requested amplified gender ratio"
        )
    return female_n, male_n, target_female_share


def sample_split(frame, n, seed, gender_amplification=1.0):
    if n > len(frame):
        raise ValueError(f"Requested {n:,} rows from a split with only {len(frame):,} rows")

    profession_counts = balanced_profession_counts(n, PROFESSION_NAMES)
    overall_female_share = frame[SENSITIVE_COLUMN].mean()
    pieces = []
    plan_rows = []

    for offset, (profession, profession_n) in enumerate(profession_counts.items()):
        profession_frame = frame.loc[frame["profession_name"].eq(profession)]
        if profession_n > len(profession_frame):
            raise ValueError(
                f"Requested {profession_n:,} rows for {profession}, "
                f"but only {len(profession_frame):,} are available"
            )

        female_n, male_n, target_female_share = gender_counts_for_profession(
            profession_frame,
            profession_n,
            overall_female_share,
            gender_amplification,
        )
        female_rows = profession_frame.loc[profession_frame[SENSITIVE_COLUMN].eq(1)]
        male_rows = profession_frame.loc[profession_frame[SENSITIVE_COLUMN].eq(0)]
        sampled_parts = []
        if female_n:
            sampled_parts.append(female_rows.sample(n=female_n, random_state=seed + offset * 2, replace=False))
        if male_n:
            sampled_parts.append(male_rows.sample(n=male_n, random_state=seed + offset * 2 + 1, replace=False))
        pieces.append(pd.concat(sampled_parts))
        plan_rows.append(
            {
                "profession": profession,
                "rows": profession_n,
                "female_rows": female_n,
                "male_rows": male_n,
                "source_female_share": profession_frame[SENSITIVE_COLUMN].mean(),
                "target_female_share": target_female_share,
                "sample_female_share": female_n / profession_n,
            }
        )

    sampled = pd.concat(pieces).sort_index().reset_index(names="source_index")
    plan = pd.DataFrame(plan_rows)
    return sampled, plan


def choose_gender_amplification(frame, n, seed):
    rows = []
    chosen = None
    for amplification in GENDER_AMPLIFICATION_GRID:
        sampled, _ = sample_split(frame, n, seed, gender_amplification=amplification)
        sample_mi = gender_profession_mutual_info(sampled)
        rows.append(
            {
                "gender_amplification": amplification,
                "sample_gender_profession_mi": sample_mi,
                "meets_target": sample_mi >= TARGET_GENDER_PROFESSION_MI,
            }
        )
        if chosen is None and sample_mi >= TARGET_GENDER_PROFESSION_MI:
            chosen = amplification
    if chosen is None:
        chosen = GENDER_AMPLIFICATION_GRID[-1]
    return chosen, pd.DataFrame(rows)


selected_gender_amplification, amplification_search = choose_gender_amplification(
    full_frames["train"],
    SAMPLE_SIZES["train"],
    RANDOM_SEED,
)

sample_frames = {}
sampling_plans = {}
for split, frame in full_frames.items():
    sample_frames[split], sampling_plans[split] = sample_split(
        frame,
        SAMPLE_SIZES[split],
        RANDOM_SEED,
        gender_amplification=selected_gender_amplification,
    )

for split, frame in sample_frames.items():
    output_path = data_dir / f"{split}.csv"
    frame.to_csv(output_path, index=False, lineterminator="\n")

sample = pd.concat(
    [frame.assign(sample_split=split) for split, frame in sample_frames.items()],
    ignore_index=True,
)

{
    "sample_rows": len(sample),
    "by_split": sample["sample_split"].value_counts().to_dict(),
    "sampling": "profession-balanced, gender-skew-amplified within profession",
    "target_gender_profession_mi": TARGET_GENDER_PROFESSION_MI,
    "selected_gender_amplification": selected_gender_amplification,
    "train_sample_gender_profession_mi": gender_profession_mutual_info(sample_frames["train"]),
    "saved_dir": str(data_dir),
    "saved_files": sorted(path.name for path in data_dir.glob("*.csv")),
}

In [ ]:
print("Amplification search on the train split")
display(
    amplification_search.style.format(
        {
            "gender_amplification": "{:.2f}",
            "sample_gender_profession_mi": "{:.4f}",
        }
    )
)

print("Per-profession sampling plan for train")
display(
    sampling_plans["train"].style.format(
        {
            "source_female_share": "{:.1%}",
            "target_female_share": "{:.1%}",
            "sample_female_share": "{:.1%}",
        }
    )
)

In [ ]:
# Sanity check for characters that can confuse editors or line-based CSV tools.
unusual_line_separators = {"\u2028": "LINE SEPARATOR", "\u2029": "PARAGRAPH SEPARATOR"}

{
    split: {
        name: int(frame[TEXT_COLUMN].str.contains(char, regex=False).sum())
        for char, name in unusual_line_separators.items()
    }
    for split, frame in sample_frames.items()
}

In [ ]:
def distribution(frame, column, normalize=False):
    counts = frame[column].value_counts(normalize=normalize).sort_index()
    if normalize:
        counts = counts.mul(100).round(2)
    return counts.rename("percent" if normalize else "count")


split_distribution = pd.DataFrame({
    split: distribution(frame, "sample_split")
    for split, frame in {"sample": sample}.items()
})

gender_counts = pd.concat(
    {split: distribution(frame, "gender_name") for split, frame in sample_frames.items()},
    axis=1,
).fillna(0).astype(int)

gender_percent = pd.concat(
    {split: distribution(frame, "gender_name", normalize=True) for split, frame in sample_frames.items()},
    axis=1,
).fillna(0)

print("Gender counts")
display(gender_counts)

print("Gender percentages")
display(gender_percent)

In [ ]:
profession_counts = pd.concat(
    {split: distribution(frame, "profession_name") for split, frame in sample_frames.items()},
    axis=1,
).fillna(0).astype(int)

profession_percent = pd.concat(
    {split: distribution(frame, "profession_name", normalize=True) for split, frame in sample_frames.items()},
    axis=1,
).fillna(0)

print("Profession counts")
display(profession_counts)

print("Profession percentages")
display(profession_percent)

In [ ]:
for split, frame in sample_frames.items():
    print(f"{split}: profession x gender counts")
    display(pd.crosstab(frame["profession_name"], frame["gender_name"]))

    print(f"{split}: female percentage by profession")
    female_pct = (
        pd.crosstab(frame["profession_name"], frame["gender_name"], normalize="index")
        .get("female", pd.Series(0.0, index=sorted(frame["profession_name"].unique())))
        .mul(100)
        .round(1)
        .sort_values(ascending=False)
        .rename("female_percent")
        .to_frame()
    )
    display(female_pct)

In [ ]:
text_length_summary = sample.groupby("sample_split")["text_length"].agg(
    rows="count",
    mean="mean",
    median="median",
    min="min",
    p10=lambda s: s.quantile(0.10),
    p90=lambda s: s.quantile(0.90),
    max="max",
).round(1)

text_length_summary

## Gender-Profession Signal After Sampling

As a final verification, compare the mutual information between the dataset gender label and profession before and after sampling. Since this notebook runs before concept labeling, this check uses the raw `gender` column rather than concept answers.

In [ ]:
def gender_profession_mutual_info(frame):
    gender = frame[[SENSITIVE_COLUMN]].to_numpy()
    profession = frame[TARGET_COLUMN].to_numpy()
    return float(
        mutual_info_classif(
            gender,
            profession,
            discrete_features=True,
            random_state=RANDOM_SEED,
        )[0]
    )


mi_rows = []
for split in SAMPLE_SIZES:
    full_mi = gender_profession_mutual_info(full_frames[split])
    sample_mi = gender_profession_mutual_info(sample_frames[split])
    mi_rows.append(
        {
            "split": split,
            "full_gender_profession_mi": full_mi,
            "sample_gender_profession_mi": sample_mi,
            "delta_sample_minus_full": sample_mi - full_mi,
        }
    )

mi_summary = pd.DataFrame(mi_rows)
mi_summary.style.format(
    {
        "full_gender_profession_mi": "{:.4f}",
        "sample_gender_profession_mi": "{:.4f}",
        "delta_sample_minus_full": "{:+.4f}",
    }
)

In [ ]:
# Compare the sampled splits against their full source splits.
def compare_sample_to_full(split):
    full = full_frames[split]
    sampled = sample_frames[split]

    profession_compare = pd.DataFrame({
        "full_percent": distribution(full, "profession_name", normalize=True),
        "sample_percent": distribution(sampled, "profession_name", normalize=True),
    }).fillna(0)
    profession_compare["delta_pct_points"] = (
        profession_compare["sample_percent"] - profession_compare["full_percent"]
    ).round(2)

    gender_compare = pd.DataFrame({
        "full_percent": distribution(full, "gender_name", normalize=True),
        "sample_percent": distribution(sampled, "gender_name", normalize=True),
    }).fillna(0)
    gender_compare["delta_pct_points"] = (
        gender_compare["sample_percent"] - gender_compare["full_percent"]
    ).round(2)

    return profession_compare, gender_compare


for split in sample_frames:
    profession_compare, gender_compare = compare_sample_to_full(split)
    print(f"{split}: gender drift from full split")
    display(gender_compare)
    print(f"{split}: largest profession drift from full split")
    display(profession_compare.reindex(profession_compare["delta_pct_points"].abs().sort_values(ascending=False).index).head(10))

## Concept Labeling


In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm

SPLITS = ["train", "validation", "test"]
TEXT_COLUMN = "hard_text"
OPENAI_MODEL = "gpt-5-nano"


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")


repo_root = find_repo_root(Path.cwd())
load_dotenv(repo_root / ".env")

concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
sample_dir = repo_root / "data" / "bias_in_bios_sample"
labels_dir = repo_root / "data" / "bias_in_bios_concept_labels"
requests_dir = labels_dir / "openai_batch_requests"
responses_dir = labels_dir / "openai_batch_responses"
metadata_path = labels_dir / "openai_batch_jobs.json"

labels_dir.mkdir(parents=True, exist_ok=True)
requests_dir.mkdir(parents=True, exist_ok=True)
responses_dir.mkdir(parents=True, exist_ok=True)

{
    "repo_root": str(repo_root),
    "concepts_path": str(concepts_path),
    "sample_dir": str(sample_dir),
    "labels_dir": str(labels_dir),
    "openai_model": OPENAI_MODEL,
    "openai_key_loaded": bool(os.environ.get("OPENAI_API_KEY")),
    "splits": SPLITS,
}

In [ ]:
concepts_df = pd.read_csv(concepts_path)
split_frames = {
    split: pd.read_csv(sample_dir / f"{split}.csv")
    for split in SPLITS
}

{
    "num_concepts": len(concepts_df),
    "split_rows": {split: len(frame) for split, frame in split_frames.items()},
    "total_rows": sum(len(frame) for frame in split_frames.values()),
}

In [ ]:
LEXICAL_PATTERNS = {
    "male_pronouns": re.compile(r"\b(he|him|his|himself)\b", re.IGNORECASE),
    "female_pronouns": re.compile(r"\b(she|her|hers|herself)\b", re.IGNORECASE),
    "male_gendered_titles": re.compile(r"\b(Mr|Mister|Sir|Lord)\.?\b"),
    "female_gendered_titles": re.compile(r"\b(Mrs|Ms|Miss|Madam|Madame|Lady|Dame)\.?\b"),
}


def regex_label_text(text):
    labels = {}
    for concept, pattern in LEXICAL_PATTERNS.items():
        match = pattern.search(text)
        labels[concept] = {
            "label": int(bool(match)),
            "evidence": match.group(0) if match else "",
            "source": "regex",
        }
    return labels


api_concepts_df = concepts_df.loc[~concepts_df["concept"].isin(LEXICAL_PATTERNS)].copy()
{"regex_concepts": list(LEXICAL_PATTERNS), "api_concepts": len(api_concepts_df)}

In [ ]:
def build_labeling_prompt(text, concepts):
    concept_lines = "\n".join(
        f"- {row.concept}: {row.description}"
        for row in concepts.itertuples(index=False)
    )

    return f"""Label biography concepts.

Rules:
- Use only the biography text.
- Concept names are shorthand; use the full description to decide each label.
- Concepts are independent and may overlap; mark every clearly supported concept as 1.
- For each concept, choose label 1 when the biography clearly supports it.
- Choose label 0 when the concept is absent or only weakly implied.
- Return only binary 0/1 labels, with no evidence text.

Concepts:
{concept_lines}

Biography:
{text}
"""


def build_response_schema(concepts):
    properties = {
        row.concept: {"type": "integer", "enum": [0, 1]}
        for row in concepts.itertuples(index=False)
    }
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "concept_labels",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties),
                "additionalProperties": False,
            },
        },
    }


response_format = build_response_schema(api_concepts_df)

In [ ]:
def parse_json_response(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def normalize_api_label(value):
    if isinstance(value, dict):
        label = value.get("label", 0)
        if isinstance(label, str):
            label = label.strip().lower() in {"1", "true", "yes"}
        return {
            "label": int(bool(label)),
            "evidence": str(value.get("evidence", "")),
            "source": "openai_batch",
        }
    return {"label": int(bool(value)), "evidence": "", "source": "openai_batch"}


def build_labels_frame(split, row, api_labels):
    regex_labels = regex_label_text(row[TEXT_COLUMN])
    all_labels = {concept: normalize_api_label(value) for concept, value in api_labels.items()}
    all_labels.update(regex_labels)

    frame = concepts_df[["concept", "kind", "description"]].copy()
    frame.insert(0, "split", split)
    frame.insert(1, "source_index", int(row["source_index"]))
    frame.insert(2, "sample_row", int(row.name))
    frame.insert(3, "profession_name", row["profession_name"])
    frame.insert(4, "gender_name", row["gender_name"])
    frame.insert(5, TEXT_COLUMN, row[TEXT_COLUMN])
    frame["label"] = frame["concept"].map(lambda name: all_labels.get(name, {}).get("label", 0)).astype(int)
    frame["evidence"] = frame["concept"].map(lambda name: all_labels.get(name, {}).get("evidence", ""))
    frame["label_source"] = frame["concept"].map(lambda name: all_labels.get(name, {}).get("source", "missing"))
    return frame

In [ ]:
def build_batch_request(split, row):
    return {
        "custom_id": f"{split}:{int(row.name)}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": OPENAI_MODEL,
            "messages": [
                {"role": "system", "content": "Return concept labels that follow the provided JSON schema."},
                {"role": "user", "content": build_labeling_prompt(row[TEXT_COLUMN], api_concepts_df)},
            ],
            "response_format": response_format,
        },
    }


def write_batch_requests(split_frames):
    request_paths = {}
    for split, frame in split_frames.items():
        path = requests_dir / f"{split}_requests.jsonl"
        rows = frame.iterrows()
        with path.open("w", encoding="utf-8") as handle:
            for _, row in tqdm(rows, total=len(frame), desc=f"Writing {split} requests"):
                handle.write(json.dumps(build_batch_request(split, row), ensure_ascii=False) + "\n")
        request_paths[split] = path
    return request_paths


request_paths = write_batch_requests(split_frames)
{split: str(path) for split, path in request_paths.items()}

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY was not loaded from .env")

client = OpenAI()
batch_jobs = {}

for split, request_path in tqdm(request_paths.items(), total=len(request_paths), desc="Submitting batch jobs"):
    input_file = client.files.create(file=request_path.open("rb"), purpose="batch")
    batch = client.batches.create(
        input_file_id=input_file.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={"split": split, "dataset": "bias_in_bios_sample"},
    )
    batch_jobs[split] = {
        "input_file_id": input_file.id,
        "batch_id": batch.id,
        "status": batch.status,
    }

metadata_path.write_text(json.dumps(batch_jobs, indent=2), encoding="utf-8")
pd.DataFrame.from_dict(batch_jobs, orient="index")

In [ ]:
batch_jobs = json.loads(metadata_path.read_text(encoding="utf-8"))
client = OpenAI()

batch_status = []
for split, job in tqdm(batch_jobs.items(), total=len(batch_jobs), desc="Checking batch jobs"):
    batch = client.batches.retrieve(job["batch_id"])
    job.update(
        {
            "status": batch.status,
            "output_file_id": batch.output_file_id,
            "error_file_id": batch.error_file_id,
        }
    )
    batch_status.append(
        {
            "split": split,
            "batch_id": batch.id,
            "status": batch.status,
            "request_counts": batch.request_counts,
            "output_file_id": batch.output_file_id,
            "error_file_id": batch.error_file_id,
        }
    )

metadata_path.write_text(json.dumps(batch_jobs, indent=2), encoding="utf-8")
pd.DataFrame(batch_status)

In [ ]:
batch_jobs = json.loads(metadata_path.read_text(encoding="utf-8"))
client = OpenAI()

for split, job in tqdm(batch_jobs.items(), total=len(batch_jobs), desc="Downloading batch outputs"):
    batch = client.batches.retrieve(job["batch_id"])
    if batch.status != "completed":
        raise RuntimeError(f"Batch for {split} is {batch.status}, not completed yet.")
    if not batch.output_file_id:
        raise RuntimeError(f"Batch for {split} completed without an output file.")

    output_path = responses_dir / f"{split}_responses.jsonl"
    client.files.content(batch.output_file_id).write_to_file(output_path)
    job["status"] = batch.status
    job["output_file_id"] = batch.output_file_id
    job["output_path"] = str(output_path)

metadata_path.write_text(json.dumps(batch_jobs, indent=2), encoding="utf-8")
{split: job["output_path"] for split, job in batch_jobs.items()}

In [ ]:
def parse_batch_response_line(line):
    item = json.loads(line)
    if item.get("error"):
        raise RuntimeError(f"Batch request failed for {item['custom_id']}: {item['error']}")

    split, sample_row_text = item["custom_id"].split(":", 1)
    sample_row = int(sample_row_text)
    message = item["response"]["body"]["choices"][0]["message"]["content"]
    return split, sample_row, parse_json_response(message)


response_paths = {
    path.name.removesuffix("_responses.jsonl"): path
    for path in responses_dir.glob("*_responses.jsonl")
}
if not response_paths:
    raise FileNotFoundError(f"No *_responses.jsonl files found in {responses_dir}")

label_frames = []
failed_rows = []

for split, output_path in tqdm(response_paths.items(), total=len(response_paths), desc="Parsing response files"):
    if split not in split_frames:
        raise ValueError(f"Response file split '{split}' is not in SPLITS={SPLITS}")

    total_rows = len(split_frames[split])
    with output_path.open("r", encoding="utf-8") as handle:
        for line in tqdm(handle, total=total_rows, desc=f"Parsing {split}", leave=False):
            try:
                response_split, sample_row, api_labels = parse_batch_response_line(line)
                row = split_frames[response_split].loc[sample_row]
                label_frames.append(build_labels_frame(response_split, row, api_labels))
            except Exception as exc:
                failed_rows.append({"split": split, "error": str(exc), "line": line[:500]})

if failed_rows:
    failed_path = labels_dir / "failed_batch_rows.jsonl"
    with failed_path.open("w", encoding="utf-8") as handle:
        for row in failed_rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    raise RuntimeError(f"Failed to parse {len(failed_rows)} batch rows. See {failed_path}.")

labels_long = pd.concat(label_frames, ignore_index=True)
parsed_splits = "_".join(response_paths)
labels_long_path = labels_dir / f"labels_long_{parsed_splits}.csv"
labels_long.to_csv(labels_long_path, index=False, lineterminator="\n")

label_matrix = labels_long.pivot_table(
    index=["split", "sample_row", "source_index", "profession_name", "gender_name"],
    columns="concept",
    values="label",
    aggfunc="max",
    fill_value=0,
).reset_index()
labels_wide_path = labels_dir / f"labels_wide_{parsed_splits}.csv"
label_matrix.to_csv(labels_wide_path, index=False, lineterminator="\n")

{
    "parsed_splits": list(response_paths),
    "labels_long_rows": len(labels_long),
    "label_matrix_rows": len(label_matrix),
    "labels_long_path": str(labels_long_path),
    "labels_wide_path": str(labels_wide_path),
}

In [ ]:
label_diagnostics = {
    "total_positive_labels": int(labels_long["label"].sum()),
    "openai_batch_positive_labels": int(labels_long.loc[labels_long["label_source"].eq("openai_batch"), "label"].sum()),
    "regex_positive_labels": int(labels_long.loc[labels_long["label_source"].eq("regex"), "label"].sum()),
    "rows_with_at_least_one_positive": int(labels_long.groupby(["split", "sample_row"])["label"].sum().gt(0).sum()),
}

if label_diagnostics["openai_batch_positive_labels"] == 0:
    raise RuntimeError("OpenAI Batch API returned zero positive labels.")

label_diagnostics

In [ ]:
concept_summary = (
    labels_long.groupby(["concept", "kind"], as_index=False)["label"]
    .agg(yes_count="sum", total="count", yes_rate="mean")
    .sort_values(["yes_count", "concept"], ascending=[False, True])
)
concept_summary["yes_rate"] = concept_summary["yes_rate"].round(3)
concept_summary

## Review Tables

Use these tables to inspect concept activity and individual examples.

In [ ]:
row_summary = (
    labels_long.groupby(["split", "sample_row", "source_index", "profession_name", "gender_name"], as_index=False)["label"]
    .sum()
    .rename(columns={"label": "num_positive_concepts"})
    .sort_values(["num_positive_concepts", "split", "sample_row"])
    .reset_index(drop=True)
)
row_summary

In [ ]:
# Change this to any row from row_summary when you want to inspect a specific biography.
review_row = row_summary.iloc[8]
REVIEW_SPLIT = review_row["split"]
REVIEW_SAMPLE_ROW = int(review_row["sample_row"])

example = split_frames[REVIEW_SPLIT].loc[REVIEW_SAMPLE_ROW]
print(f"split={REVIEW_SPLIT}, sample_row={REVIEW_SAMPLE_ROW}, profession={example['profession_name']}, gender={example['gender_name']}")
print(example[TEXT_COLUMN])

labels_long.loc[
    labels_long["split"].eq(REVIEW_SPLIT) & labels_long["sample_row"].eq(REVIEW_SAMPLE_ROW),
    ["concept", "kind", "label", "evidence", "label_source"],
].sort_values(["label", "concept"], ascending=[False, True])

In [ ]:
label_matrix.head()

In [ ]:
{
    "labels_long_path": str(labels_long_path),
    "labels_wide_path": str(labels_wide_path),
    "batch_jobs_path": str(metadata_path),
}

In [ ]:
# Final verification before using the labels for Concept-QA training.
wide_labels = pd.read_csv(labels_wide_path)
concept_columns = concepts_df["concept"].tolist()
expected_split_rows = {split: len(frame) for split, frame in split_frames.items()}
expected_total_rows = sum(expected_split_rows.values())

missing_concepts = sorted(set(concept_columns) - set(wide_labels.columns))
unexpected_concepts = sorted(
    set(wide_labels.columns)
    - {"split", "sample_row", "source_index", "profession_name", "gender_name"}
    - set(concept_columns)
)
if missing_concepts or unexpected_concepts:
    raise RuntimeError(
        f"Concept columns mismatch. Missing={missing_concepts}, unexpected={unexpected_concepts}"
    )

split_counts = wide_labels["split"].value_counts().reindex(SPLITS, fill_value=0).to_dict()
if split_counts != expected_split_rows:
    raise RuntimeError(f"Split row counts mismatch. Expected={expected_split_rows}, actual={split_counts}")

if len(wide_labels) != expected_total_rows:
    raise RuntimeError(f"Expected {expected_total_rows} rows, found {len(wide_labels)}")

if wide_labels.duplicated(["split", "sample_row"]).any():
    duplicate_count = int(wide_labels.duplicated(["split", "sample_row"]).sum())
    raise RuntimeError(f"Found {duplicate_count} duplicate split/sample_row pairs")

for split, frame in split_frames.items():
    actual_rows = sorted(wide_labels.loc[wide_labels["split"].eq(split), "sample_row"].tolist())
    expected_rows = list(frame.index)
    if actual_rows != expected_rows:
        raise RuntimeError(f"sample_row values do not align for split={split}")

non_binary_values = sorted(set(wide_labels[concept_columns].stack().dropna().unique()) - {0, 1})
if non_binary_values:
    raise RuntimeError(f"Found non-binary concept labels: {non_binary_values}")

concept_positive_counts = wide_labels[concept_columns].sum().astype(int)
zero_concepts = concept_positive_counts[concept_positive_counts.eq(0)].index.tolist()
if zero_concepts:
    raise RuntimeError(f"Concepts with zero positive labels: {zero_concepts}")

per_split_positive_counts = wide_labels.groupby("split")[concept_columns].sum().astype(int)
rows_with_any_positive = wide_labels[concept_columns].sum(axis=1).gt(0)

verification_summary = {
    "labels_wide_path": str(labels_wide_path),
    "num_rows": len(wide_labels),
    "expected_rows": expected_total_rows,
    "split_counts": split_counts,
    "num_concepts": len(concept_columns),
    "zero_positive_concepts": zero_concepts,
    "rows_with_at_least_one_positive": int(rows_with_any_positive.sum()),
    "rows_with_zero_positive": int((~rows_with_any_positive).sum()),
    "min_concept_positive_count": int(concept_positive_counts.min()),
    "max_concept_positive_count": int(concept_positive_counts.max()),
}

verification_summary